In [1]:
import ee
import requests
import zipfile
import pandas as pd
from google.colab import auth, files

# 1. Initialize
auth.authenticate_user()
PROJECT_ID = 'satelite-514107'
ee.Initialize(project=PROJECT_ID)

# 2. AOI: Greater Lahore Bounding Box (~1803 sq km)
lahore = ee.Geometry.Rectangle([74.10, 31.30, 74.55, 31.68])

# 3. Preprocessing Function (Spectral Bands + NDVI + NDBI)
def prep_s2(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return img.select(['B2', 'B3', 'B4', 'B8', 'B11']).addBands([ndvi, ndbi])

# Filter Sentinel-2 (Autumn dry season: Oct-Nov)
s2_2019 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(lahore).filterDate('2019-10-01', '2019-11-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(prep_s2).median().clip(lahore)

s2_2025 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(lahore).filterDate('2025-10-01', '2025-11-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(prep_s2).median().clip(lahore)

# 4. Ground Truth Training & Validation Split (80% Train / 20% Validation)
veg_geom = ee.Geometry.MultiPolygon([
    [[[74.18, 31.32], [74.22, 31.32], [74.22, 31.35], [74.18, 31.35]]],
    [[[74.45, 31.50], [74.48, 31.50], [74.48, 31.53], [74.45, 31.53]]]
])
built_geom = ee.Geometry.MultiPolygon([
    [[[74.32, 31.51], [74.36, 31.51], [74.36, 31.54], [74.32, 31.54]]],
    [[[74.28, 31.46], [74.31, 31.46], [74.31, 31.49], [74.28, 31.49]]]
])
other_geom = ee.Geometry.MultiPolygon([
    [[[74.26, 31.60], [74.30, 31.60], [74.30, 31.63], [74.26, 31.63]]]
])

veg_samples = s2_2019.sample(region=veg_geom, scale=20, numPixels=120, geometries=True).map(lambda f: f.set('landcover', 0))
built_samples = s2_2019.sample(region=built_geom, scale=20, numPixels=120, geometries=True).map(lambda f: f.set('landcover', 1))
other_samples = s2_2019.sample(region=other_geom, scale=20, numPixels=80, geometries=True).map(lambda f: f.set('landcover', 2))

all_samples = veg_samples.merge(built_samples).merge(other_samples).randomColumn('random', seed=42)
training_data = all_samples.filter(ee.Filter.lt('random', 0.8))
validation_data = all_samples.filter(ee.Filter.gte('random', 0.8)).limit(50)

# 5. Train Random Forest (100 Trees)
features = ['B2', 'B3', 'B4', 'B8', 'B11', 'NDVI', 'NDBI']
rf_model = ee.Classifier.smileRandomForest(100).train(
    features=training_data,
    classProperty='landcover',
    inputProperties=features
)

class_2019 = s2_2019.classify(rf_model)
class_2025 = s2_2025.classify(rf_model)

# 6. Validation Matrix & CSV Export (50 Points)
validated = validation_data.classify(rf_model)
error_matrix = validated.errorMatrix('landcover', 'classification')
val_acc = error_matrix.accuracy().getInfo() * 100
print(f"Validation Accuracy (50 Points): {val_acc:.2f}%")

val_features = validated.getInfo()['features']
val_rows = []
for i, feat in enumerate(val_features, 1):
    props = feat['properties']
    coords = feat['geometry']['coordinates']
    actual = props['landcover']
    pred = props['classification']
    val_rows.append({
        "Point_ID": i,
        "Longitude": round(coords[0], 5),
        "Latitude": round(coords[1], 5),
        "Actual_Class": actual,
        "Predicted_Class": pred,
        "Status": "Correct" if actual == pred else "Incorrect"
    })
pd.DataFrame(val_rows).to_csv("validation_points_50.csv", index=False)

# 7. Change Detection: 2019 Vegetation (0) -> 2025 Built-up (1)
conversion_binary = class_2019.eq(0).And(class_2025.eq(1))
conversion_layer = conversion_binary.selfMask().rename('converted')

# 8. Smog Layer (Sentinel-5P Aerosol Index)
smog_layer = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_AER_AI') \
    .filterBounds(lahore).filterDate('2025-10-15', '2025-11-15') \
    .select('absorbing_aerosol_index').mean().clip(lahore)

# 9. Compute Area & Smog Statistics
pixel_area = ee.Image.pixelArea().divide(1e6)

total_aoi_sqkm = lahore.area().divide(1e6).getInfo()
veg_2019_sqkm = pixel_area.updateMask(class_2019.eq(0)).reduceRegion(ee.Reducer.sum(), lahore, 30, maxPixels=1e9).get('area').getInfo()
built_2025_sqkm = pixel_area.updateMask(class_2025.eq(1)).reduceRegion(ee.Reducer.sum(), lahore, 30, maxPixels=1e9).get('area').getInfo()
converted_sqkm = pixel_area.updateMask(conversion_binary).reduceRegion(ee.Reducer.sum(), lahore, 30, maxPixels=1e9).get('area').getInfo()
loss_pct = (converted_sqkm / veg_2019_sqkm) * 100  # % of 2019 farmland/vegetation converted

mean_aai = smog_layer.reduceRegion(ee.Reducer.mean(), lahore, 1000, maxPixels=1e9).get('absorbing_aerosol_index').getInfo()
mean_aai = round(mean_aai, 2) if mean_aai is not None else 0.0
smog_status = "Severe Plume" if mean_aai >= 1.5 else ("Moderate Haze" if mean_aai >= 0.8 else "Low / Clear")

# 10. Print Ready-to-Paste Backend Dictionaries
rgb_vis = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}
ndvi_vis = {'min': -0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']}
ndbi_vis = {'min': -0.5, 'max': 0.5, 'palette': ['green', 'white', 'red']}
class_vis = {'min': 0, 'max': 2, 'palette': ['228B22', '808080', 'EEDC82']}
change_vis = {'min': 1, 'max': 1, 'palette': ['FF0000']}
smog_vis = {'min': -0.5, 'max': 2.5, 'palette': ['black', 'blue', 'cyan', 'green', 'yellow', 'red']}

print("\n--- PASTE INTO backend/services/change_service.py ---")
print(f"""CACHED_METRICS = {{
    "aoi": "Lahore District",
    "total_area_sqkm": {round(total_aoi_sqkm, 2)},
    "vegetation_2019_sqkm": {round(veg_2019_sqkm, 2)},
    "builtup_2025_sqkm": {round(built_2025_sqkm, 2)},
    "converted_sqkm": {round(converted_sqkm, 2)},
    "loss_pct": {round(loss_pct, 2)},
    "smog_status": "{smog_status}",
    "mean_aerosol_index": {mean_aai}
}}""")

print("\n--- PASTE INTO backend/services/gee_service.py ---")
print(f"""CACHED_TILES = {{
    "rgb_2019": "{s2_2019.getMapId(rgb_vis)['tile_fetcher'].url_format}",
    "rgb_2025": "{s2_2025.getMapId(rgb_vis)['tile_fetcher'].url_format}",
    "class_2019": "{class_2019.getMapId(class_vis)['tile_fetcher'].url_format}",
    "class_2025": "{class_2025.getMapId(class_vis)['tile_fetcher'].url_format}",
    "change": "{conversion_layer.getMapId(change_vis)['tile_fetcher'].url_format}",
    "smog": "{smog_layer.getMapId(smog_vis)['tile_fetcher'].url_format}"
}}""")

# 11. Zip and Download All 10 PNG Maps + Validation CSV
all_maps = [
    (s2_2019, rgb_vis, "rgb_2019.png"),
    (s2_2025, rgb_vis, "rgb_2025.png"),
    (s2_2019.select('NDVI'), ndvi_vis, "ndvi_2019.png"),
    (s2_2025.select('NDVI'), ndvi_vis, "ndvi_2025.png"),
    (s2_2019.select('NDBI'), ndbi_vis, "ndbi_2019.png"),
    (s2_2025.select('NDBI'), ndbi_vis, "ndbi_2025.png"),
    (class_2019, class_vis, "classification_2019.png"),
    (class_2025, class_vis, "classification_2025.png"),
    (conversion_layer, change_vis, "change_detection_map.png"),
    (smog_layer, smog_vis, "smog_aerosol_map.png")
]

with zipfile.ZipFile("geoguardian_outputs.zip", "w") as zipf:
    zipf.write("validation_points_50.csv")
    for img, vis, filename in all_maps:
        url = img.getThumbURL({**vis, 'dimensions': 1024, 'region': lahore, 'format': 'png'})
        img_data = requests.get(url).content
        with open(filename, 'wb') as f:
            f.write(img_data)
        zipf.write(filename)
        print(f"Zipped: {filename}")

files.download("geoguardian_outputs.zip")

ModuleNotFoundError: No module named 'google.colab'